# Loss landscapes around truth (HK multi-wavelength closure)

Pairs of fit parameters against each other, as in the LUCiD calibration slides: each axis is a
parameter's scale relative to truth, the colour is the local Δloss (log scale), and the white
streamlines run down the gradient towards the panel's minimum. The loss is the closure fit's own:
the cross χ² (Σ r_A·r_B over two independent forward draws, so no Monte-Carlo variance term),
gains profiled as the fit does, every other parameter at truth. Truth is the star, the fit the
cross, its path the thin line.

Settings (pairs, spans, grid, rays) are constants at the top of `hk_landscape_job.py`; it reads the
closure run's results file, so point `--results` at the run to map. This notebook only loads and plots.

In [ ]:
import sys

# sensor_R against every other parameter of the 337 nm single-wavelength run (PAIRS in the job).
# Check nvidia-smi for an idle GPU first.
GPU = 1
!cd .. && {sys.executable} hk_landscape_job.py --gpu {GPU} --results results/hk_tabulated_resume.npz --out results/hk_landscape_337.npz

In [ ]:
import sys

# sensor_R alone at 337 nm, gains fixed at truth vs profiled (hk_sensor_profile_job.py); builds its
# own truth, no results file. A different idle GPU from the landscape, to run both at once.
GPU = 0
!cd .. && {sys.executable} hk_sensor_profile_job.py --gpu {GPU} --out results/hk_sensor_profile_337.npz

In [ ]:
import sys

# Fisher check: WCSim's rgcff (table rebuilt, R + A + T = 1) in place of the scale sensor_R, 6 wavelengths,
# 5 diffusers. Prints sigma / correlations for sensor_R, rgcff and both. Check nvidia-smi first.
GPU = 0
!cd .. && {sys.executable} hk_rgcff_fisher.py --gpu {GPU} --out results/hk_rgcff_fisher.npz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

L = np.load('../results/hk_landscape_337.npz')
names, truth = list(L['names']), L['truth']
idx = {n: i for i, n in enumerate(names)}
fit = L['fit'] if 'fit' in L.files else None
hist = L['history'] if 'history' in L.files else None
pairs = [tuple(p) for p in L['pairs']]
label = lambda n: n.replace('_', ' ')

deltas = [g - g.min() for g in L['loss']]
positive = np.concatenate([d[d > 0] for d in deltas])
norm = LogNorm(vmin=np.percentile(positive, 1), vmax=positive.max())

ncol = min(3, len(pairs)) if len(pairs) != 4 else 2
nrow = int(np.ceil(len(pairs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.6 * ncol, 4.8 * nrow), squeeze=False,
                         layout='constrained')
print(f'{"pair":44s}{"minimum at (x, y) scale":>26s}{"loss(truth) - min":>20s}')
for ax, (a, b), sa, sb, grid, d in zip(axes.flat, pairs, L['scale_x'], L['scale_y'], L['loss'], deltas):
    mesh = ax.pcolormesh(sa, sb, np.clip(d, norm.vmin, None), norm=norm, cmap='viridis',
                         shading='gouraud')
    gy, gx = np.gradient(grid, sb, sa)
    ax.streamplot(sa, sb, -gx, -gy, color='white', density=1.1, linewidth=0.7, arrowsize=0.8)
    if hist is not None:
        ax.plot(hist[:, idx[a]] / truth[idx[a]], hist[:, idx[b]] / truth[idx[b]],
                color='cyan', lw=0.8, alpha=0.7)
    ax.plot(1, 1, marker='*', ms=15, color='deeppink', mec='white', ls='', label='truth')
    if fit is not None:
        ax.plot(fit[idx[a]] / truth[idx[a]], fit[idx[b]] / truth[idx[b]], marker='X', ms=11,
                color='cyan', mec='k', ls='', label='fit')
    iy, ix = np.unravel_index(np.argmin(grid), grid.shape)
    ax.plot(sa[ix], sb[iy], marker='o', ms=5, color='white', ls='', label='panel minimum')
    ax.set_xlim(sa[0], sa[-1]); ax.set_ylim(sb[0], sb[-1])
    ax.set_xlabel(f'{label(a)} scale'); ax.set_ylabel(f'{label(b)} scale')
    ax.set_title(f'{label(a)} vs {label(b)}')
    i1 = np.argmin(np.abs(sa - 1)); j1 = np.argmin(np.abs(sb - 1))
    print(f'{a + " x " + b:44s}{f"({sa[ix]:.3f}, {sb[iy]:.3f})":>26s}{grid[j1, i1] - grid.min():20.1f}')
for ax in axes.flat[len(pairs):]:
    ax.set_visible(False)
axes.flat[0].legend(loc='upper left', fontsize=8, framealpha=0.7)
fig.colorbar(mesh, ax=axes, label='local Δloss (cross χ²)', shrink=0.8)
fig.suptitle(f'loss landscape around truth ({L["results"]}, {int(L["n_rays"]):,} rays)', fontweight='bold')
plt.show()

In [ ]:
# sensor_R scan at 337 nm: gains fixed at truth vs profiled as the fit does
P = np.load('../results/hk_sensor_profile_337.npz')
s = P['scale']
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), layout='constrained')

ax = axes[0]
for key, lab in (('cross_fix', 'gains fixed at truth'), ('cross_prof', 'gains profiled')):
    c = P[key]
    ax.plot(s, c - c[np.argmin(np.abs(s - 1))], marker='o', label=lab)
ax.set_ylabel('cross χ² − value at sensor_R = 1'); ax.set_title('loss along sensor_R')

ax = axes[1]
for key, lab in (('g_fix', 'gains fixed at truth'), ('g_prof', 'gains profiled')):
    ax.errorbar(s, P[key], yerr=P[key + '_sem'], marker='o', capsize=3, label=lab)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('gradient Jᵀr along sensor_R'); ax.set_title('gradient (zero = stopping point)')

ax = axes[2]
ax.plot(s, s * np.exp(P['step']), marker='o', label='where the fit steps to')
ax.plot(s, s, color='grey', ls=':', label='no move')
ax2 = ax.twinx()
ax2.plot(s, 100 * P['dk_rms'], color='C3', marker='s', label='profiled gain shift RMS')
ax2.set_ylabel('RMS(k / k_true − 1) (%)', color='C3')
ax.set_ylabel('sensor_R after one damped step'); ax.set_title('the fit’s own update, profiled gains')

for ax in axes:
    ax.axvline(1, color='deeppink', lw=1, ls='--')
    ax.set_xlabel('sensor_R (× truth)'); ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.show()

In [ ]:
import sys

# The 395 nm Asimov closure (results/hk_395nm.npz): five pairs (PAIRS in the job).
# Check nvidia-smi for an idle GPU first.
GPU = 1
!cd .. && {sys.executable} hk_landscape_job.py --gpu {GPU} --results results/hk_395nm.npz --out results/hk_landscape_395.npz

In [ ]:
# 395 nm Asimov closure landscapes: 3 + 2 panels, truth star, fit cross, fit path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import LogNorm

L = np.load('../results/hk_landscape_395.npz')
names, truth = list(L['names']), L['truth']
idx = {n: i for i, n in enumerate(names)}
fit = L['fit'] if 'fit' in L.files else None
hist = L['history'] if 'history' in L.files else None
pairs = [tuple(p) for p in L['pairs']]
label = lambda n: n.replace('_395', '').replace('_', ' ')

deltas = [g - g.min() for g in L['loss']]
positive = np.concatenate([d[d > 0] for d in deltas])
norm = LogNorm(vmin=np.percentile(positive, 1), vmax=positive.max())

ncol = 3
nrow = int(np.ceil(len(pairs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.6 * ncol, 4.8 * nrow), squeeze=False,
                         layout='constrained')
print(f'{"pair":44s}{"minimum at (x, y) scale":>26s}{"loss(truth) - min":>20s}')
for ax, (a, b), sa, sb, grid, d in zip(axes.flat, pairs, L['scale_x'], L['scale_y'], L['loss'], deltas):
    mesh = ax.pcolormesh(sa, sb, np.clip(d, norm.vmin, None), norm=norm, cmap='viridis',
                         shading='gouraud')
    gy, gx = np.gradient(grid, sb, sa)
    ax.streamplot(sa, sb, -gx, -gy, color='white', density=1.1, linewidth=0.7, arrowsize=0.8)
    if hist is not None:
        # The fit's path: bright orange with a dark outline, to stand out on both ends of viridis.
        ax.plot(hist[:, idx[a]] / truth[idx[a]], hist[:, idx[b]] / truth[idx[b]],
                color='#ff8c00', lw=2.2, label='fit path', zorder=4,
                path_effects=[pe.Stroke(linewidth=3.6, foreground='k'), pe.Normal()])
    ax.plot(1, 1, marker='*', ms=15, color='deeppink', mec='white', ls='', label='truth', zorder=5)
    if fit is not None:
        ax.plot(fit[idx[a]] / truth[idx[a]], fit[idx[b]] / truth[idx[b]], marker='X', ms=11,
                color='cyan', mec='k', ls='', label='fit', zorder=6)
    iy, ix = np.unravel_index(np.argmin(grid), grid.shape)
    ax.plot(sa[ix], sb[iy], marker='o', ms=5, color='white', ls='', label='panel minimum', zorder=5)
    ax.set_xlim(sa[0], sa[-1]); ax.set_ylim(sb[0], sb[-1])
    ax.set_xlabel(f'{label(a)} (× truth)'); ax.set_ylabel(f'{label(b)} (× truth)')
    ax.set_title(f'{label(a)} vs {label(b)}')
    i1 = np.argmin(np.abs(sa - 1)); j1 = np.argmin(np.abs(sb - 1))
    print(f'{a + " x " + b:44s}{f"({sa[ix]:.3f}, {sb[iy]:.3f})":>26s}{grid[j1, i1] - grid.min():20.1f}')
for ax in axes.flat[len(pairs):]:
    ax.set_visible(False)
axes.flat[0].legend(loc='upper left', fontsize=8, framealpha=0.7)
fig.colorbar(mesh, ax=axes, label='local Δloss (cross χ²)', shrink=0.8)
fig.suptitle('395 nm closure: loss landscape around truth', fontweight='bold')
plt.show()